# E09: Suelo, ráster y profundidad

[Enunciado](../../talleres/E09.md) · [Entorno WSL](../../docs/instalacion-wsl.md)

Ejecutar en orden con el kernel del curso. Preparar las fuentes indicadas en el enunciado; no se descargan automáticamente. Las salidas se regeneran en `kit/salidas/`. Los originales se conservan.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import geopandas as gpd
from IPython.display import display, Image
candidatos = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'big-data-postgraduate']
REPO = next((p for p in candidatos if (p/'kit/talleres.py').exists()), None)
assert REPO is not None, 'Abrir desde el repositorio o su carpeta base bigdata'
KIT = REPO/'kit'
sys.path.insert(0, str(KIT))
import talleres as t
import geoespacial as geo
RAW, OUT = t.RAW, t.OUT


## 1. Inventario y mapa
SoilGrids y WoSIS son suficientes para el comando suelo. La comparación temática posterior requiere los dos archivos IGAC indicados en E09.

In [ ]:
control=t.suelo()
assert tuple(control['soilgrids']['shape'])==(42,44)
assert control['soilgrids']['celdas_cero']==188
assert control['soilgrids']['pixeles_validos']==1660
display(control)
display(Image(filename=str(OUT/'mapa_suelo.png')))

## 2. Perfiles y horizontes
Diez registros no significan diez sitios: distinguir perfil de horizonte. Revisar método, profundidad e incertidumbre de ubicación antes de comparar con una predicción.

In [ ]:
wosis=geo.leer_vector('wosis.json')
assert len(wosis)==10 and wosis.profile_id.nunique()==3
display(wosis[['profile_id','upper_depth','lower_depth','value_avg','method_options','positional_uncertainty']].sort_values(['profile_id','upper_depth']))
import rasterio
from shapely.geometry import box
with rasterio.open(RAW/'soilgrids.tif') as src:
    dentro=wosis.to_crs(src.crs).intersects(box(*src.bounds))
print('Horizontes dentro del recorte:',int(dentro.sum()))

## 3. Capas temáticas y límites de interpretación
Mostrar los atributos de química y correlación para distinguir intervalos cartográficos y taxonomía. No calcular error de predicción sin coincidencia espacial, profundidad y método compatibles.

In [ ]:
capa=geo.leer_vector('igac_quimica.json')
print('IGAC química: CRS',capa.crs,'extensión',capa.total_bounds)
display(capa.drop(columns='geometry'))
# Correlación es una tabla ArcGIS de atributos, no una capa con geometría.
correlacion=json.loads((RAW/'igac_correlacion.json').read_text())
tabla_correlacion=pd.DataFrame([f['attributes'] for f in correlacion['features']])
assert len(tabla_correlacion)==10
display(tabla_correlacion)


**Entrega:** mapa, control_suelo.json y tabla por perfil/profundidad. Los 188 ceros no tienen NoData declarado: se separan provisionalmente para la media, conservando el original. El factor de escala es valor/10. Documentar las condiciones que faltan para validar la predicción; no interpretar el recorte como análisis de finca.